In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import h5py

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
repo_dir = Path("../../..")
data_dir = repo_dir / "extract_n_eval" / "data"
save_dir = repo_dir / "visualization" / "paper" / "supp" / "figures"

assert data_dir.exists(), f"data directory {data_dir} does not exist!"

# Helpers

In [ ]:
def load_noise_ceilings(file_path, sqrt_nc=False):
    """Load noise ceilings from an HDF5 file."""

    noise_ceiling_dict = {}
    with h5py.File(file_path, "r") as f:
        subjects = f.attrs["subjects"]
        rois = f.attrs["rois"]
        time_points = None
        if "time_points" in f.attrs:
            time_points = f.attrs["time_points"]
        
        for subj in tqdm(subjects, desc="Subjects", leave=False):
            noise_ceiling_dict[subj] = {}
            for roi in tqdm(rois, desc="ROIs", leave=False):
                nc = f[f"noise_ceilings/{subj}/{roi}"][:]
                if sqrt_nc:
                    nc = np.sqrt(nc/100) * 100
                noise_ceiling_dict[subj][roi] = nc
                
    return noise_ceiling_dict, subjects, rois, time_points


def print_noise_ceilings(nc_dict, subjects, rois):
    """Print noise ceilings in a readable format."""
    for subj in subjects:
        print(f"Subject: {subj}")
        for roi in rois:
            nc = nc_dict[subj][roi]
            if len(nc) == 0:
                text = f"  ROI: {roi}, No data available."
            else:
            
                text = (
                    f"  ROI: {roi}, Noise Ceiling: {nc.mean():.4f} ± {nc.std():.4f}, "
                    f"min:{nc.min():.4f}, max:{nc.max():.4f}, Shape: {nc.shape}"
                )
            print(text)
        print()
                


def convert_to_dataframe(nc_dict, subjects, rois, time_points=None):
    """Convert noise ceilings dictionary to a pandas DataFrame."""
    records = []
    if time_points is not None:
        # Datasets with time dimension, e.g., EEG and MEG
        for subj in subjects:
            for roi in rois:
                nc_values = nc_dict[subj][roi]
                for idx, t in enumerate(time_points):
                    val = nc_values[:, idx].mean() # Averaging across channels/sensors
                    records.append({"subject": subj, "roi": roi, "time": t, "nc": val})
        
    else:
        # Non-time datasets, e.g., fMRI and electrophysiology
        for subj in subjects:
            for roi in rois:
                nc_values = nc_dict[subj][roi]
                for value in nc_values:
                    records.append({"subject": subj, "roi": roi, "nc": value})
    
    df = pd.DataFrame.from_records(records)
    return df

In [ ]:

def plot_hist(
    df: pd.DataFrame,
    x: str,
    hue: str,
    rows: int,
    cols: int,
    roi_list: list,
    sub_fig_size: tuple = (6, 4),
    dpi: int = 300,
    zoom: float = 1.0,
    bins: int = None,
    stat: str = "density",
    kde: bool = False,
    color: str = "C1",
):
    figsize = (cols * sub_fig_size[0]*zoom, rows * sub_fig_size[1]*zoom)
    fig, axes = plt.subplots(rows, cols, figsize=figsize, dpi=dpi)
    
    axes = axes.flatten() if rows * cols > 1 else [axes]
    
    assert len(roi_list) <= len(axes), (
        f"Number of subplots ({len(axes)}) is less than number of unique ROIs ({len(roi_list)}). "
        "Please increase rows/cols or reduce the number of ROIs."
    )
    
    if bins is None:
        bins = np.linspace(10, 100, 30)
    
    for i, roi in enumerate(roi_list):
        ax = axes[i]
        sub_df = df[df['roi'] == roi]
        sns.histplot(
            sub_df,
            x=x,
            stat=stat,
            hue=hue,
            bins=bins,
            kde=kde,
            ax=ax,
            color=color,
        )

        ax.set_title(roi, fontsize=16, fontweight='bold')
        ax.set_xlabel("Noise Ceiling", fontsize=12, fontweight='bold')
        ax.set_ylabel(f"{stat.capitalize()}", fontsize=12, fontweight='bold')
        ax.spines[['top', 'right']].set_visible(False)
        ax.legend_.set_title(hue.capitalize())
        
        if i != len(roi_list) - 1:
            ax.get_legend().remove()
            
            
    for j in range(i + 1, len(axes)):
        axes[j].remove()
        
        
    plt.tight_layout()
    return fig, axes


In [ ]:
def plot_timeseries(
    df: pd.DataFrame,
    hue: str,
    rows: int,
    cols: int,
    roi_list: list,
    ylim: tuple,
    sub_fig_size: tuple = (6, 4),
    dpi: int = 300,
    zoom: float = 1.0,
    color: str = "C1",
):
    figsize = (cols * sub_fig_size[0]*zoom, rows * sub_fig_size[1]*zoom)
    fig, axes = plt.subplots(rows, cols, figsize=figsize, dpi=dpi)
    
    axes = axes.flatten() if rows * cols > 1 else [axes]
    
    assert len(roi_list) <= len(axes), (
        f"Number of subplots ({len(axes)}) is less than number of ROIs({len(roi_list)}). "
        "Please increase rows/cols or reduce the number of ROIs."
    )
    
    for i, roi in enumerate(roi_list):
        ax = axes[i]
        sub_df = df[df['roi'] == roi]
        sns.lineplot(
            sub_df,
            x="time",
            y="nc",
            hue=hue,
            ax=ax,
            color=color,
        )

        ax.set_title(roi.capitalize(), fontsize=16, fontweight='bold')
        ax.set_xlabel("Time (s)", fontsize=12, fontweight='bold')
        ax.set_ylabel(f"Noise Ceiling", fontsize=12, fontweight='bold')
        ax.set_ylim(ylim)
        ax.spines[['top', 'right']].set_visible(False)
        ax.legend_.set_title(hue.capitalize())
        
        if i != len(roi_list) - 1:
            ax.get_legend().remove()
            
            
    for j in range(i + 1, len(axes)):
        axes[j].remove()
        
        
    plt.tight_layout()
    return fig, axes


In [ ]:
def save_figs(fig, save_dir, base_filename, dpi=300, formats=("png", "pdf", "svg")):
    for fmt in formats:
    
        save_fmt_dir = Path(save_dir) / fmt
        if not save_fmt_dir.exists():
            save_fmt_dir.mkdir(parents=True, exist_ok=False)
        
        file_path = save_fmt_dir / f"{base_filename}.{fmt}"
        
        fig.savefig(file_path, dpi=dpi, bbox_inches='tight')

        print(f"Figure saved to {file_path}")

# FreemanZiembaV1V2

In [ ]:

dataset_path = data_dir / "bs_fz.h5"

fz_nc, fz_subjects, fz_rois, _ = load_noise_ceilings(dataset_path)

print_noise_ceilings(fz_nc, fz_subjects, fz_rois)


fz_df = convert_to_dataframe(fz_nc, fz_subjects, fz_rois)

fig, axes = plot_hist(
    df=fz_df,
    x="nc",
    hue="subject",
    stat="count",
    rows=1,
    cols=2,
    # zoom=0.75,
    zoom=1,
    sub_fig_size=(6, 3),
    roi_list=fz_rois,
)

axes[-1].legend_.remove()

save_figs(
    fig,
    save_dir=save_dir,
    base_filename="noise_ceilings_histograms-bs_fz",
    dpi=300,
    formats=("png", "pdf", "svg"),
)


In [ ]:
# fig, axes = plt.subplots(1, 1, figsize=(8, 6), dpi=300)

# ax = axes

# # sns.histplot(fz_nc["monkeys"]["V1"], bins=30, kde=True, ax=ax, color="C0")
# sns.scatterplot(y=fz_nc["monkeys"]["V1"], x=np.arange(len(fz_nc["monkeys"]["V1"])), ax=ax)

# MajajHongV4IT

In [ ]:
dataset_path = data_dir / "bs_mh.h5"

mh_nc, mh_subjects, mh_rois, _ = load_noise_ceilings(dataset_path)

mh_df = convert_to_dataframe(mh_nc, mh_subjects, mh_rois)

print_noise_ceilings(mh_nc, mh_subjects, mh_rois)



mh_df = convert_to_dataframe(mh_nc, mh_subjects, mh_rois)

fig, axes = plot_hist(
    df=mh_df,
    x="nc",
    hue="subject",
    stat="count",
    rows=1,
    cols=2,
    zoom=1,
    sub_fig_size=(6, 3),
    roi_list=mh_rois,
)

# axes[-1].legend_.remove()

save_figs(
    fig,
    save_dir=save_dir,
    base_filename="noise_ceilings_histograms-bs_mh",
    dpi=300,
    formats=("png", "pdf", "svg"),
)

In [ ]:
# fig, axes = plt.subplots(1, 1, figsize=(8, 6), dpi=300)

# ax = axes

# sns.histplot(mh_df, x="nc", stat="density", bins=30, kde=True, ax=ax, hue="roi", color="C1")

# TVSD

In [ ]:
dataset_path = data_dir / "tvsd.h5"

tvsd_nc, tvsd_subjects, tvsd_rois, _ = load_noise_ceilings(dataset_path)

tvsd_df = convert_to_dataframe(tvsd_nc, tvsd_subjects, tvsd_rois)


print_noise_ceilings(tvsd_nc, tvsd_subjects, tvsd_rois)


tvsd_df = convert_to_dataframe(tvsd_nc, tvsd_subjects, tvsd_rois)


# print(tvsd_df.nc.min()) # 77
bins = np.linspace(70, 100, 30)

fig, axes = plot_hist(
    df=tvsd_df,
    x="nc",
    hue="subject",
    stat="count",
    bins=bins,
    rows=1,
    cols=3,
    zoom=0.75,
    sub_fig_size=(6, 4),
    roi_list=tvsd_rois,
)

# axes[-1].legend_.remove()

save_figs(
    fig,
    save_dir=save_dir,
    base_filename="noise_ceilings_histograms-tvsd",
    dpi=300,
    formats=("png", "pdf", "svg"),
)

# NSD

In [ ]:
dataset_path = data_dir / "nsd_func1pt8mm_individualROIs.h5"

nsd_func_nc, nsd_func_subjects, nsd_func_rois, _ = load_noise_ceilings(dataset_path)

nsd_func_df = convert_to_dataframe(nsd_func_nc, nsd_func_subjects, nsd_func_rois)

fig, axes = plot_hist(
    df=nsd_func_df,
    x="nc",
    hue="subject",
    stat="count",
    rows=7,
    cols=4,
    zoom=0.75,
    roi_list=nsd_func_rois,
)

axes[-1].set_title("Whole Brain", fontsize=16, fontweight='bold')

save_figs(
    fig,
    save_dir=save_dir,
    base_filename="noise_ceilings_histograms-nsd_func1pt8mm_individualROIs",
    dpi=300,
    formats=("png", "pdf", "svg"),
)

## Compare NSD ROIs noise ceilings on fsaverage vs native surface

#### TODO: Currently, conversion to density is across all fMRI spaces. It should be done separately for fsaverage, native surface and func1pt8mm spaces.

In [ ]:
# dataset_path = data_dir / "nsd_fsaverage_individualROIs.h5"
# nsd_fs_nc, nsd_fs_subjects, nsd_fs_rois, _ = load_noise_ceilings(dataset_path)
# nsd_fs_df = convert_to_dataframe(nsd_fs_nc, nsd_fs_subjects, nsd_fs_rois)


# dataset_path = data_dir / "nsd_nativesurface_individualROIs.h5"
# nsd_native_nc, nsd_native_subjects, nsd_native_rois, _ = load_noise_ceilings(dataset_path)
# nsd_native_df = convert_to_dataframe(nsd_native_nc, nsd_native_subjects, nsd_native_rois)


# nrows, ncols = 7, 4
# sub_fig_size = (6, 4)
# zoom = 0.5
# figsize = (ncols * sub_fig_size[0] * zoom, nrows * sub_fig_size[1] * zoom)

# fig, axes = plt.subplots(nrows, ncols, figsize=figsize, dpi=300)

# df_fs_copy = nsd_fs_df.copy()
# df_func_copy = nsd_func_df.copy()
# df_native_copy = nsd_native_df.copy()

# df_fs_copy['source'] = 'fsaverage'
# df_func_copy['source'] = 'functional'
# df_native_copy['source'] = 'native surface'

# df_nsd_all = pd.concat([df_fs_copy, df_func_copy, df_native_copy], ignore_index=True)


# for i, ax in enumerate(axes.flatten()):
#     roi = nsd_fs_rois[i]
    
#     bins = np.linspace(10, 100, 30)
    
    
#     sns.histplot(df_nsd_all[df_nsd_all['roi'] == roi], x="nc", stat="density", bins=bins, kde=False, ax=ax, hue="source")
#     ax.set_title(roi, fontsize=16, fontweight='bold')
#     ax.set_xlabel("Noise Ceiling", fontsize=12, fontweight='bold')
#     ax.set_ylabel("Density", fontsize=12, fontweight='bold')
#     ax.spines[['top', 'right']].set_visible(False)
    
#     if i != len(nsd_fs_rois) - 1:
#         ax.get_legend().remove()
    
# fig.tight_layout()
# plt.show()
    

# THINGS-fMRI

In [ ]:
dataset_path = data_dir / "things_fmri.h5"

things_fmri_nc, things_fmri_subjects, things_fmri_rois, _ = load_noise_ceilings(dataset_path)

things_fmri_df = convert_to_dataframe(things_fmri_nc, things_fmri_subjects, things_fmri_rois)

things_fmri_rois = things_fmri_rois.tolist()[1:-1] + ["whole_brain"]

fig, axes = plot_hist(
    df=things_fmri_df,
    x="nc",
    hue="subject",
    stat="count",
    rows=7,
    cols=4,
    zoom=0.75,
    roi_list=things_fmri_rois,
)


axes[-1].set_title("Whole Brain", fontsize=16, fontweight='bold')

save_figs(
    fig,
    save_dir=save_dir,
    base_filename="noise_ceilings_histograms-things_fmri",
    dpi=300,
    formats=("png", "pdf", "svg"),
)

# THINGS-EEG1

In [ ]:
dataset_path = data_dir / "things_eeg1.h5"

things_eeg1_nc, things_eeg1_subjects, things_eeg1_rois, things_eeg1_time_points = load_noise_ceilings(dataset_path)

things_eeg1_df = convert_to_dataframe(things_eeg1_nc, things_eeg1_subjects, things_eeg1_rois, time_points=things_eeg1_time_points)


fig, axes = plot_timeseries(
    df=things_eeg1_df,
    hue="subject",
    rows=2,
    cols=4,
    zoom=0.75,
    roi_list=things_eeg1_rois,
    ylim=(0, 100)
)


axes.flatten()[-3].set_title("Occipital+Parietal", fontsize=16, fontweight='bold')
axes.flatten()[-2].set_title("Whole Brain", fontsize=16, fontweight='bold')
    
handles, labels = axes.flatten()[-2].get_legend_handles_labels()
labels = [f"Subject {l.split('-')[-1]}" for l in labels]
axes.flatten()[-2].legend(handles, labels, title="Subjects", bbox_to_anchor=(1.05, 1.2), loc='upper left')


save_figs(
    fig,
    save_dir=save_dir,
    base_filename="noise_ceilings_timeseries-things_eeg1",
    dpi=300,
    formats=("png", "pdf", "svg"),
)

# THINGS-EEG2

In [ ]:
dataset_path = data_dir / "things_eeg2.h5"

things_eeg2_nc, things_eeg2_subjects, things_eeg2_rois, things_eeg2_time_points = load_noise_ceilings(dataset_path)

things_eeg2_df = convert_to_dataframe(things_eeg2_nc, things_eeg2_subjects, things_eeg2_rois, time_points=things_eeg2_time_points)

fig, axes = plot_timeseries(
    df=things_eeg2_df,
    hue="subject",
    rows=2,
    cols=4,
    zoom=0.75,
    roi_list=things_eeg2_rois,
    ylim=(0, 100)
)
axes.flatten()[-3].set_title("Occipital+Parietal", fontsize=16, fontweight='bold')
axes.flatten()[-2].set_title("Whole Brain", fontsize=16, fontweight='bold')
    
handles, labels = axes.flatten()[-2].get_legend_handles_labels()
labels = [f"Subject {l.split('-')[-1]}" for l in labels]
axes.flatten()[-2].legend(handles, labels, title="Subjects", bbox_to_anchor=(1.05, 1.2), loc='upper left')


save_figs(
    fig,
    save_dir=save_dir,
    base_filename="noise_ceilings_timeseries-things_eeg2",
    dpi=300,
    formats=("png", "pdf", "svg"),
)

# THINGS-MEG

In [ ]:
dataset_path = data_dir / "things_meg.h5"

things_meg_nc, things_meg_subjects, things_meg_rois, things_meg_time_points = load_noise_ceilings(dataset_path)

things_meg_df = convert_to_dataframe(things_meg_nc, things_meg_subjects, things_meg_rois, time_points=things_meg_time_points)


fig, axes = plot_timeseries(
    df=things_meg_df,
    hue="subject",
    rows=2,
    cols=4,
    zoom=0.75,
    roi_list=things_meg_rois,
    ylim=(0, 100)
)


axes.flatten()[-3].set_title("Occipital+Parietal", fontsize=16, fontweight='bold')
axes.flatten()[-2].set_title("Whole Brain", fontsize=16, fontweight='bold')
    
handles, labels = axes.flatten()[-2].get_legend_handles_labels()
labels = [f"Subject {l.split('-')[-1]}" for l in labels]
axes.flatten()[-2].legend(handles, labels, title="Subjects", bbox_to_anchor=(1.05, 0.7), loc='upper left')



save_figs(
    fig,
    save_dir=save_dir,
    base_filename="noise_ceilings_timeseries-things_meg",
    dpi=300,
    formats=("png", "pdf", "svg"),
)